# F1 Strategy Board: EDA walkthrough

**Mapping off-track operations to on-track performance.** This notebook walks through the six steps of the EDA engine in `src/eda/`, with the finding from each chart written out beneath it.

> **Data provenance.** Results, laps, pit stops, tyre stints, standings and weather are **real** (Jolpica and FastF1). Financial, logistics and simulation / CFD data are **SYNTHETIC**: they come from the seeded generator in `src/synthetic.py`, which plants loose relationships with the real results. Findings that involve them show the method working; they are not evidence about real teams.

Built from **152 races** (2019-2025), **86** with lap-level data, using the **full download**. The notebook is generated by `notebooks/build_notebook.py`, so the findings always match the data.

## 0. Setup

Load the cleaned, joined tables built by layers 1 to 3 (`src/sources.py`, `src/ingest.py`, `src/clean.py`).

In [1]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import pandas as pd
import plotly.io as pio
from src import config, feedback, insights, model, pipeline
from src.eda import bivariate, factor_columns, outliers, reduction, report, timeseries, univariate

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", 40, "display.width", 200)

tables = pipeline.load()
tr, ts, dr = tables.team_race, tables.team_season, tables.driver_race
laps = tables.laps.sample(min(len(tables.laps), 25_000), random_state=config.SEED)   # keeps the notebook light
factors = factor_columns(tr)
print(f"Data: {'full download' if tables.meta['mode'] == 'raw' else 'bundled sample'} | seasons {min(tables.meta['seasons'])}-{max(tables.meta['seasons'])} | "
      f"{tables.meta['races']} races, {tables.meta['races_with_laps']} with lap data")
pd.DataFrame({name: frame.shape for name, frame in tables.frames().items()}, index=["rows", "columns"]).T

Data: full download | seasons 2019-2025 | 152 races, 86 with lap data


,rows,columns
races,152,23
driver_race,3038,68
team_race,1520,80
team_season,70,43
component,13680,12
laps,94817,36
stints,4673,15
pitstops,5405,16


## 1. Univariate profiling

Start with one variable at a time. For each of the four headline variables (lap time, spend, pit delta, tyre wear) the questions are the same: where is the middle, how wide is the spread, is the distribution lopsided (skew) and how heavy are its tails (kurtosis)? The answers decide which statistics are safe to use later.

### Lap time

In [2]:
result = univariate.lap_time_distribution(laps)
result.fig.show()

**Finding.** Across 94,817 laps the raw distribution is extremely right-skewed (skew 68.3): pit, safety-car and incident laps form a long slow tail. Keeping only the 66,549 clean laps (70%) leaves a tight band with a spread of 1.27% around the median.

### Spend by component (SYNTHETIC)

In [3]:
result = univariate.spend_distribution(tables.component)
result.fig.show()

**Finding.** SYNTHETIC data. Aero takes the largest share of spend (29%), followed by power unit (19%). Every component is right-skewed (overall skew 4.2): most rounds are routine and a few carry upgrade bursts.

### Pit delta

In [4]:
result = univariate.pit_delta_distribution(tables.pitstops)
result.fig.show()

**Finding.** The median is 0.00s against a mean of 0.95s. The distribution is strongly right-skewed (skew 3.83) with heavy tails with frequent extremes (excess kurtosis 20.56). Use the median, not the mean, as the typical value. 12% of stops lose more than 3s to the field: slow stops are rare but costly.

### Tyre wear

In [5]:
result = univariate.wear_distribution(tables.stints)
result.fig.show()

**Finding.** Median fuel-corrected degradation in seconds per lap: soft +0.049, medium +0.053, hard +0.049, intermediate +0.001. Values are noisy stint to stint (spread 0.078s/lap) because traffic, tyre management and track evolution all leak into the slope.

### Shape of every factor

In [6]:
result = univariate.shape_overview(tr, factors)
result.fig.show()

**Finding.** 11 of 29 variables are strongly skewed (|skew| >= 1); the most lopsided is customs delay at 7.9. Those are the ones where rank-based statistics (median, Spearman) are safer than means and Pearson correlations.

## 2. Bivariate and multivariate analysis

Now pairs and groups of variables. The heatmap gives the overview; the scatters test the two cross-domain questions in the brief, spend against reliability and weather against pace.

### Correlation heatmap

In [7]:
result = bivariate.correlation_heatmap(tr, factors + ['points', 'dnf'])
result.fig.show()

**Finding.** Spearman correlations. Strongest positive pair: wind-tunnel runs and CFD items (r = 0.94). Strongest negative pair: race pace vs field and points scored (r = -0.78). Red cells move together, cyan cells move in opposite directions, dark cells are unrelated.

### Spend vs reliability

In [8]:
result = bivariate.spend_vs_reliability(ts)
result.fig.show()

**Finding.** Spend is SYNTHETIC. Across 70 team-seasons, season spend and finish rate show no meaningful link (Spearman 0.10, p = 0.43). Inside the cost-cap era (2021 on), where budgets are compressed, it is 0.42. Bubble size is points; for comparison, spend against points gives Spearman 0.44.

### Weather vs pace

In [9]:
result = bivariate.weather_vs_pace(tables.races, tr)
result.fig.show()

**Finding.** Across 77 dry races there is no statistically clear link between track temperature and tyre degradation (Spearman -0.11, p = 0.36). Circuit layout, compound choice and tyre management vary alongside temperature and swamp any direct effect at race level.

### Wet vs dry races

In [10]:
result = bivariate.wet_vs_dry(tables.races)
result.fig.show()

**Finding.** Rain-affected races average 18.3% retirements (9 races) against 12.8% in the dry (77 races); Mann-Whitney p = 0.16, so with this many wet races the gap could still be chance. Weather shifts the risk for the whole field at once rather than for one team.

### Pair plot

In [11]:
result = bivariate.pair_plot(ts, ['spend_total_musd', 'sim_hours', 'pit_delta_s', 'dnf_rate', 'total_points'])
result.fig.show()

**Finding.** Each panel plots one pair of variables; tight diagonal bands mean a strong relationship. The tightest pair here is simulator hours and total points (Spearman 0.94).

## 3. Outlier and anomaly detection

Three detectors with different blind spots (IQR fences, z-score, Isolation Forest) vote on every team-race row. Retirements and component failures, the sport's own anomalies, are summarised separately.

In [12]:
anomaly_cols = ['spend_total_musd', 'pit_median_s', 'customs_delay_h', 'late_parts', 'cfd_items_raw', 'crew_fatigue_idx']
scored = outliers.detect_anomalies(tr, anomaly_cols, by=['season'])
scored = outliers.annotate_watchlist(scored, feedback.load_watchlist())   # feedback loop from the dashboard
scored.loc[scored['anomaly'], ['team_race_key', 'race_name', 'anomaly_votes', 'anomaly_reason']].sort_values('anomaly_votes', ascending=False).head(10)

,team_race_key,race_name,anomaly_votes,anomaly_reason
4,2019-01|mercedes,Australian Grand Prix,3,Total spend per round (USD M)
54,2019-06|mercedes,Monaco Grand Prix,3,Total spend per round (USD M)
70,2019-08|alfa,French Grand Prix,3,Late parts (count)
80,2019-09|alfa,Austrian Grand Prix,3,"CFD items, as logged"
159,2019-16|williams,Russian Grand Prix,3,Late parts (count)
215,2020-01|mercedes,Austrian Grand Prix,3,Total spend per round (USD M)
222,2020-02|ferrari,Styrian Grand Prix,3,Median pit-lane time (s)
225,2020-02|mercedes,Styrian Grand Prix,3,Total spend per round (USD M)
188,2019-19|toro_rosso,United States Grand Prix,3,"Customs delay (h), Late parts (count)"
333,2020-13|haas,Emilia Romagna Grand Prix,3,Late parts (count)


### Detector agreement

In [13]:
result = outliers.detector_agreement(scored)
result.fig.show()

**Finding.** IQR fences flag 289 rows and the z-score rule 82: the z-score is the stricter test because extreme values inflate the very standard deviation it divides by. 89 rows are flagged by at least two detectors.

### Anomaly map

In [14]:
result = outliers.anomaly_scatter(scored, 'customs_delay_h', 'pit_median_s')
result.fig.show()

**Finding.** 87 of 1497 team-race rows (5.8%) are flagged by at least two of the three detectors. Agreement between methods matters more than any single flag.

### Non-finishes by cause

In [15]:
result = outliers.dnf_by_cause(dr)
result.fig.show()

**Finding.** Retirement rates run from 10.2% (2024) to 16.6% (2022) of entries. 124 retirements have a published mechanical cause; 159 are logged only as 'Retired', so the mechanical count is a lower bound.

### Failures by component

In [16]:
result = outliers.failures_by_component(dr)
result.fig.show()

**Finding.** Power unit is the most common failed system with 47 of 124 mechanical retirements (38%).

### Retirement rate by team and season

In [17]:
result = outliers.dnf_heatmap(dr)
result.fig.show()

**Finding.** Williams has the highest average retirement rate (20%). The single worst team-season is Williams in 2021 at 25% of entries.

## 4. Time series and trends

Two clocks matter: the season (round by round, year by year) for spend and results, and the stint (lap by lap of tyre age) for degradation.

### Spend by season (SYNTHETIC)

In [18]:
result = timeseries.multi_season_trend(ts, 'spend_total_musd')
result.fig.show()

**Finding.** From 2019 to 2025 season spend fell for every team: most for Mercedes (-307.61) and least for Williams (-38.32). SYNTHETIC: the generator applies the published cap, so the spread between teams collapses from 2021.

### Points by season

In [19]:
result = timeseries.multi_season_trend(ts, 'total_points')
result.fig.show()

**Finding.** From 2019 to 2025 the biggest rise in points is McLaren (+688.00) and the biggest fall is Mercedes (-270.00).

### Spend vs points within a season

In [20]:
result = timeseries.spend_vs_performance(tr, int(tr['season'].max()))
result.fig.show()

**Finding.** Spend is SYNTHETIC. Both lines are cumulative shares of the team's own season total. On average the spend line sits +4 percentage points relative to the points line (positive means money goes out before results come in). The widest gap is RB (+12 percentage points).

### Does spend lead or lag points?

In [21]:
result = timeseries.spend_points_lag(ts)
result.fig.show()

**Finding.** Spend is SYNTHETIC. Season spend lines up most closely with previous season's points (Spearman 0.60). That is the generator's design: budgets follow prize money from the year before. It is also the textbook reverse-causality trap: points fund spend as much as spend buys points.

### Tyre degradation curves

In [22]:
result = timeseries.tyre_degradation_curves(tables.laps)
result.fig.show()

**Finding.** Fitted degradation: soft +0.021s/lap, medium +0.022s/lap, hard +0.030s/lap, intermediate -0.013s/lap. The hard compound fades fastest. Curves that dip below zero early show tyres coming up to temperature and the track rubbering in before wear takes over. Fuel correction assumes 0.055s per lap of fuel.

### Pit-stop performance

In [23]:
result = timeseries.pit_trend(tables.pitstops)
result.fig.show()

**Finding.** Below zero is faster than the field. Over the selected seasons Red Bull has the quickest typical stop (-0.37s vs the field) and Haas the slowest (+0.54s). The whole spread between best and worst is 0.91s.

## 5. Dimensionality reduction and clustering

Thirteen operational variables per team-season are compressed with PCA, embedded with t-SNE, and grouped into operating profiles with k-means. The number of profiles is chosen by silhouette score.

In [24]:
pca = reduction.run_pca(ts, reduction.PROFILE_FEATURES)
clusters = reduction.kmeans_profiles(ts, reduction.PROFILE_FEATURES)
coords, method = reduction.embed(ts, reduction.PROFILE_FEATURES, 'tsne')
ts_clustered = ts.assign(cluster=clusters.labels)
reduction.cluster_membership(ts, clusters)

,cluster,members,avg_points,avg_position,examples
0,Profile 1,46,78.4,7.2,"McLaren 2019, Renault 2019, Toro Rosso 2019, R..."
1,Profile 2,24,524.2,2.2,"Mercedes 2019, Ferrari 2019, Red Bull 2019, Me..."


### PCA: variance explained

In [25]:
result = reduction.pca_scree(pca)
result.fig.show()

**Finding.** The first two components capture 59% of the variance in 13 features, and 5 components are needed to pass 80%. The operational variables are heavily redundant: a few underlying dimensions drive most of them.

### PCA biplot

In [26]:
result = reduction.pca_biplot(pca, ts_clustered, color='cluster', hover='constructor_name')
result.fig.show()

**Finding.** PC1 is driven mainly by sim-to-track correlation, simulator hours, spares coverage; PC2 by total spend per round, customs delay, crew fatigue index. Points that sit close together operate alike; arrows show which way each feature pulls.

### t-SNE embedding

In [27]:
result = reduction.embedding_plot(ts_clustered, coords, method, color='team', hover='constructor_name')
result.fig.show()

**Finding.** t-SNE places rows with similar operating numbers next to each other. Only neighbourhoods are meaningful: the axes have no units and distances between far-apart groups should not be read. Tight single-colour islands mean a team operates in a consistent, distinctive way.

### Choosing k

In [28]:
result = reduction.silhouette_plot(clusters)
result.fig.show()

**Finding.** k = 2 gives the best silhouette score (0.340). On a scale where 1 is perfectly separated and 0 is overlapping, the profiles are reasonably distinct: treat them as a useful summary, not as hard categories.

### Operating profiles

In [29]:
result = reduction.cluster_heatmap(clusters)
result.fig.show()

**Finding.** Red is above the overall average, cyan below, in standard deviations. Profile 1: low spares coverage and low sim-to-track correlation; Profile 2: high spares coverage and high sim-to-track correlation.

## 6. Automated EDA report

Everything above is packaged by `src/eda/report.py` into a single interactive HTML file, plus Markdown and JSON versions of the findings. The same function backs the dashboard's *Report* tab.

In [30]:
path = report.generate_report(tables)
print('Report written to', path.relative_to(ROOT))
print((config.REPORTS_DIR / 'eda_findings.md').read_text(encoding='utf-8')[:1200])

Report written to reports\eda_report.html
# F1 Strategy Board - EDA findings

Generated 2026-10-10 11:14 UTC. Data mode: raw.

> Financial, logistics and simulation / CFD data are SYNTHETIC.


## Univariate profiling

- **Lap Time Distribution**: Across 94,817 laps the raw distribution is extremely right-skewed (skew 68.3): pit, safety-car and incident laps form a long slow tail. Keeping only the 66,549 clean laps (70%) leaves a tight band with a spread of 1.27% around the median.
- **Spend By Component**: SYNTHETIC data. Aero takes the largest share of spend (29%), followed by power unit (19%). Every component is right-skewed (overall skew 4.2): most rounds are routine and a few carry upgrade bursts.
- **Pit Delta Distribution**: The median is 0.00s against a mean of 0.95s. The distribution is strongly right-skewed (skew 3.83) with heavy tails with frequent extremes (excess kurtosis 20.56). Use the median, not the mean, as the typical value. 12% of stops lose more than 3s to the field

### Where the EDA leads: ranked factors

The EDA feeds layers 5 and 6. Factors are ranked by the consensus of four methods (Spearman, partial correlation controlling for last season's strength, standardised regression and gradient-boosting permutation importance).

In [31]:
rankings = insights.build_all(tr, ts)
rankings['points'][['consensus_rank', 'label', 'domain', 'origin', 'spearman', 'partial', 'score']].head(10)

,consensus_rank,label,domain,origin,spearman,partial,score
0,1,Race pace vs field (%),Telemetry,REAL,-0.784432,-0.601705,0.982143
1,2,Mean grid slot,Telemetry,REAL,-0.772851,-0.552789,0.982143
2,3,Simulator hours,Simulation / CFD,SYNTHETIC,0.656579,0.232876,0.910714
3,4,Sim-to-track correlation (%),Simulation / CFD,SYNTHETIC,0.629234,0.377398,0.839286
4,5,Spares coverage (%),Logistics,SYNTHETIC,0.493583,0.259466,0.821429
5,6,Set-up iterations,Simulation / CFD,SYNTHETIC,0.458309,0.176157,0.741071
6,7,Pit time vs field (s),Pit stops,REAL,-0.304629,-0.178153,0.732143
7,8,Aero testing allocation (%),Simulation / CFD,SYNTHETIC,-0.525495,0.054245,0.705357
8,9,Late parts (count),Logistics,SYNTHETIC,-0.255915,-0.140322,0.687500
9,10,Total spend per round (USD M),Financial,SYNTHETIC,0.232777,0.081046,0.660714


**Finding.** For points per race:

- **#1 Race pace vs field (%)** (Telemetry, REAL): higher values go with fewer points per race (Spearman -0.78). After controlling for last season's strength the link holds up (partial -0.60).
- **#2 Mean grid slot** (Telemetry, REAL): higher values go with fewer points per race (Spearman -0.77). After controlling for last season's strength the link holds up (partial -0.55).
- **#3 Simulator hours** (Simulation / CFD, SYNTHETIC): higher values go with more points per race (Spearman +0.66). After controlling for last season's strength the link mostly disappears (partial +0.23). This factor is SYNTHETIC: the link was planted by the generator.

In [32]:
insights.recovery_check(tr)[['factor', 'target', 'planted', 'observed_r', 'recovered']]

,factor,target,planted,observed_r,recovered
0,Total spend per round (USD M),Points scored,+,0.232777,True
1,Aero share of spend,Points scored,+,0.242630,True
2,Upgrades brought,Points scored,+,0.027530,False
3,Late parts (count),Retirements,+,0.214393,True
4,Spares coverage (%),Retirements,-,-0.191789,True
5,Crew fatigue index,Pit time vs field (s),+,0.310272,True
6,Sim-to-track correlation (%),Mean grid slot,-,-0.718867,True
7,Aero testing allocation (%),Points scored,-,-0.525495,True


**Finding.** 7 of 8 relationships planted in the synthetic data are recovered with the correct sign. Because the synthetic sources have a known ground truth, this is the appropriate test for them: it checks the engine, not the teams.

## Summary of findings

The last finding from each step:

- **Univariate profiling**: 11 of 29 variables are strongly skewed (|skew| >= 1); the most lopsided is customs delay at 7.9. Those are the ones where rank-based statistics (median, Spearman) are safer than means and Pearson correlations.
- **Bivariate and multivariate analysis**: Each panel plots one pair of variables; tight diagonal bands mean a strong relationship. The tightest pair here is simulator hours and total points (Spearman 0.94).
- **Outlier and anomaly detection**: Williams has the highest average retirement rate (20%). The single worst team-season is Williams in 2021 at 25% of entries.
- **Time series and trends**: Below zero is faster than the field. Over the selected seasons Red Bull has the quickest typical stop (-0.37s vs the field) and Haas the slowest (+0.54s). The whole spread between best and worst is 0.91s.
- **Dimensionality reduction and clustering**: Red is above the overall average, cyan below, in standard deviations. Profile 1: low spares coverage and low sim-to-track correlation; Profile 2: high spares coverage and high sim-to-track correlation.

### Caveats

- Three of the five sources are synthetic, so every result that involves budget, logistics or simulation reflects the generator's design.
- Everything here is association, not causation: points fund budgets as much as budgets buy points.
- Retirement causes are not published from 2023, so component-level failure analysis covers 2019 to 2022.
- Weather is shared by every team in a race, so it explains field-wide effects, not gaps between teams.